# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Lectura guiada · Machine Learning UTP
Cristian Alvarado · Ejecuta de arriba hacia abajo. No hay tareas ni entregas. Los ejemplos se revisan con el docente.

## 1. K-Means: agrupar fincas
Seis medidas de fincas simuladas se escalan antes de calcular distancias. k es el número de grupos solicitado. Revisa codo y silhouette sin asumir un k universal. La proyección PCA facilita ver los grupos; interprétalos con las medidas originales.

In [ ]:
from IPython.display import display
# ================================================================
# CLUSTERING CON K-MEANS (Colab listo para pegar)
# Caso didáctico: segmentación de fincas agrícolas por características
# ================================================================
# ¿Qué hace este notebook?
# 1) Genera un dataset sintético "realista" de fincas con variables agro.
# 2) Normaliza (es crucial para K-means).
# 3) Selecciona k con método del codo (inertia) y silhouette.
# 4) Entrena K-means y proyecta a 2D con PCA para visualizar.
# 5) Interpreta clústeres: centroides en unidades originales y “perfiles”.
# 6) Muestra cómo asignar un nuevo ejemplo al grupo más cercano.
#
# ------------------------------------------------
# ¿Cómo funciona K-means? (resumen algorítmico)
# ------------------------------------------------
# - Entrada: datos numéricos y un número de clústeres k.
# - Inicialización: elige k centroides (p. ej. k-means++).
# - Paso de asignación: asigna cada punto al centroide más cercano (distancia euclídea).
# - Paso de actualización: recalcula cada centroide como la media de los puntos de su clúster.
# - Repite asignar/actualizar hasta converger (cambios mínimos o máx. iteraciones).
# - Objetivo: minimizar la suma de distancias cuadráticas dentro de cada clúster (inertia).
#
# Interpretación práctica:
# - Cada clúster agrupa observaciones similares; el centroide resume el “perfil medio”.
# - Elegir k: buscamos equilibrio entre compacidad (inertia baja) y separación (silhouette alta).
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ----------------------------
# 1) Generación del dataset
# ----------------------------
# Variables (todas continuas):
# - lluvia_mm                (anual)
# - temp_media_c             (°C)
# - ph_suelo                 (0-14)
# - materia_organica_pct     (%)
# - altitud_m                (msnm)
# - pendiente_pct            (%)
#
# Creamos 3 “tipos” de fincas con medias diferentes para que K-means los detecte.

rng = np.random.default_rng(7)

n1, n2, n3 = 150, 160, 140  # tamaños por grupo
# Grupo 1: tierras bajas, clima cálido, más materia orgánica
g1 = np.column_stack([
    rng.normal(1800, 180, n1),  # lluvia_mm
    rng.normal(25.0, 1.2, n1),  # temp_media_c
    rng.normal(6.3, 0.3, n1),   # ph_suelo
    rng.normal(5.5, 0.8, n1),   # materia_organica_pct
    rng.normal(600, 120, n1),   # altitud_m
    rng.normal(4, 1.5, n1)      # pendiente_pct
])

# Grupo 2: altiplano templado, pH algo mayor, pendiente media
g2 = np.column_stack([
    rng.normal(1400, 160, n2),
    rng.normal(19.0, 1.0, n2),
    rng.normal(6.8, 0.25, n2),
    rng.normal(3.8, 0.6, n2),
    rng.normal(1500, 200, n2),
    rng.normal(8, 2.0, n2)
])

# Grupo 3: alta montaña fresca, suelos más ácidos y pendientes altas
g3 = np.column_stack([
    rng.normal(1100, 150, n3),
    rng.normal(15.0, 1.1, n3),
    rng.normal(5.8, 0.25, n3),
    rng.normal(3.2, 0.5, n3),
    rng.normal(2400, 220, n3),
    rng.normal(15, 3.0, n3)
])

X = np.vstack([g1, g2, g3])
df = pd.DataFrame(X, columns=[
    "lluvia_mm", "temp_media_c", "ph_suelo",
    "materia_organica_pct", "altitud_m", "pendiente_pct"
])

print("Dimensiones del dataset:", df.shape)
display(df.head())

# ----------------------------
# 2) Normalización
# ----------------------------
# K-means usa distancia euclídea; si no escalamos, una variable con unidades grandes (p. ej. altitud)
# dominará el agrupamiento. Usamos StandardScaler (media 0, var 1).
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# ----------------------------
# 3) Selección de k
# ----------------------------
# a) Método del codo: graficamos inertia vs. k y buscamos el “codo”.
# b) Silhouette: valores más altos (~0.5-1) indican mejor separación entre clústeres.

k_values = range(2, 9)
inertias = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=7)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(6,4))
plt.plot(k_values, inertias, marker='o')
plt.title("Método del codo (Inertia vs k)")
plt.xlabel("k")
plt.ylabel("Inertia (suma de distancias al cuadrado)")
plt.grid(True)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(k_values, silhouettes, marker='o')
plt.title("Índice de Silhouette vs k")
plt.xlabel("k")
plt.ylabel("Silhouette")
plt.grid(True)
plt.show()

print("Tabla k / inertia / silhouette")
print(pd.DataFrame({"k": k_values, "inertia": inertias, "silhouette": silhouettes}))

# Tip: para este dataset, esperamos k≈3 (porque lo generamos así),
# pero en datos reales NO sabemos k; por eso miramos ambas curvas y contexto de negocio.

# ----------------------------
# 4) Ajuste con k elegido para este ejemplo (aquí tomamos k=3)
# ----------------------------
k_opt = 3
kmeans = KMeans(n_clusters=k_opt, n_init=10, random_state=7)
labels = kmeans.fit_predict(X_scaled)

df_clusters = df.copy()
df_clusters["cluster"] = labels

print("\nConteo por clúster")
print(df_clusters["cluster"].value_counts().sort_index())

# ----------------------------
# 5) Proyección 2D con PCA para visualizar clústeres
# ----------------------------
# PCA no “mejora” el clustering; solo reduce a 2D para ver separación aproximada.
pca = PCA(n_components=2, random_state=7)
X_2d = pca.fit_transform(X_scaled)
centroids_2d = pca.transform(kmeans.cluster_centers_)

plt.figure(figsize=(6,5))
for c in range(k_opt):
    plt.scatter(X_2d[labels==c, 0], X_2d[labels==c, 1], s=18, alpha=0.7, label=f"Cluster {c}")
plt.scatter(centroids_2d[:,0], centroids_2d[:,1], s=180, marker='X', edgecolor='k', label="Centroides")
plt.title("Clústeres en 2D (PCA)")
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.legend()
plt.grid(True)
plt.show()

# ----------------------------
# 6) Interpretación de clústeres
# ----------------------------
# a) Centroides en espacio escalado -> difícil de leer.
# b) Convertimos centroides a unidades originales con inverse_transform.
centroids_original = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=df.columns
).round(2)
centroids_original.index = [f"Cluster {i}" for i in range(k_opt)]

print("\nCentroides en unidades originales (interpretables):")
display(centroids_original)

# Perfil de cada clúster: medias por variable (debería ser similar a los centroides)
perfil = df_clusters.groupby("cluster").mean(numeric_only=True).round(2)
perfil.index = [f"Cluster {i}" for i in perfil.index]
print("\nPerfil (media por variable y clúster):")
display(perfil)

# Para resaltar variables más “diferenciadoras” por clúster,
# calculamos z-scores de los centroides vs. media global.
mu = df.mean()
sigma = df.std(ddof=0)

z_centroids = ((centroids_original - mu) / sigma).round(2)
print("\nZ-scores de centroides (positivo = por encima del promedio global):")
display(z_centroids)

# Guía rápida de lectura:
print("""
================= CÓMO LEER LOS RESULTADOS =================
1) Gráfica PCA:
   - Nubes separadas = clústeres bien diferenciados.
   - El marcador X indica el centroide (promedio) de cada grupo.

2) Centroides en unidades originales:
   - Interpretación directa. Ejemplo: si 'Cluster 2' tiene 'altitud_m' = 2300,
     ese grupo agrupa fincas de alta montaña.

3) Z-scores de centroides:
   - Valores > 0: variable por encima del promedio global del dataset en ese clúster.
   - Valores < 0: por debajo del promedio.
   - Útiles para describir “rasgos distintivos” de cada segmento.

4) Reglas prácticas:
   - Si un clúster muestra lluvia alta + temp alta + baja altitud,
     podrías asociarlo a tierras bajas cálidas.
   - Si otro clúster tiene pH más ácido + alta altitud + alta pendiente,
     podrías asociarlo a alta montaña con suelos más ácidos, etc.

5) ¿k correcto?
   - Revisa el “codo” (inertia) y el valor de silhouette.
   - Confirma con criterio de negocio: ¿tiene sentido operativo diferenciar esos grupos?
=============================================================
""")

# ----------------------------
# 7) Asignar un nuevo ejemplo a un grupo
# ----------------------------
# Proceso: (i) escalar con el MISMO scaler, (ii) usar kmeans.predict, (iii) interpretar.
nuevo = pd.DataFrame([{
    "lluvia_mm": 1350,
    "temp_media_c": 18.5,
    "ph_suelo": 6.9,
    "materia_organica_pct": 3.7,
    "altitud_m": 1600,
    "pendiente_pct": 7.0
}])

nuevo_scaled = scaler.transform(nuevo)
cluster_pred = kmeans.predict(nuevo_scaled)[0]

print("Nuevo ejemplo:")
display(nuevo)
print(f"Asignado a: Cluster {cluster_pred}")

print("""
Interpretación:
- Compara las variables del nuevo ejemplo con los centroides en unidades originales.
- Describe el perfil del grupo asignado en esta simulación; su número es un identificador.
- Una decisión de manejo requeriría datos representativos y evaluación agronómica adicional.
""")


## 2. Mean-Shift: zonas densas
Este bloque usa otra simulación: no compara exactamente las mismas fincas del bloque anterior. bandwidth controla la búsqueda; no pedir k no elimina los parámetros. Las etiquetas son identificadores, no niveles de calidad. DBSCAN se añade en el laboratorio complementario.

In [ ]:
from IPython.display import display
# =====================================================================
# CLUSTERING CON MEAN-SHIFT (Colab listo para pegar)
# Caso didáctico: segmentación de fincas agrícolas por características
# =====================================================================
# ¿Qué hace este notebook?
# 1) Genera un dataset sintético "realista" de fincas con variables agro.
# 2) Normaliza (crítico para distancias).
# 3) Estima automáticamente el parámetro clave: bandwidth (radio del kernel).
# 4) Ajusta Mean-Shift (encuentra k dinámicamente, sin fijar k).
# 5) Visualiza en 2D (PCA), muestra centros y tamaños de clúster.
# 6) Interpreta: centroides en unidades originales y z-scores.
# 7) “Predice” nuevos puntos por asignación al centro más cercano (ver nota).
#
# ---------------------------------------------------------------------
# ¿Cómo funciona Mean-Shift? (resumen algorítmico)
# ---------------------------------------------------------------------
# - La implementación usada de scikit-learn utiliza un kernel plano: calcula
#   la media de los puntos dentro de una vecindad de radio bandwidth.
# - "Desplaza" cada punto gradualmente hacia regiones de mayor densidad
#   (modos de la distribución). Puntos que convergen al mismo modo forman un clúster.
# - No requiere k. La cantidad de clústeres emerge de la densidad y del bandwidth.
#
# Trade-offs:
# - Bandwidth pequeño => muchos clústeres (sobre-segmentación).
# - Bandwidth grande  => pocos clústeres (sub-segmentación).
# - Complejidad alta en datasets grandes; conviene escalar y muestrear para estimar bandwidth.
# =====================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.metrics import silhouette_score

# ----------------------------
# 1) Generación del dataset
# ----------------------------
# Variables continuas:
# - lluvia_mm, temp_media_c, ph_suelo, materia_organica_pct, altitud_m, pendiente_pct
# Creamos 3 “regímenes” para que Mean-Shift pueda detectarlos sin fijar k.

rng = np.random.default_rng(11)
n1, n2, n3 = 150, 160, 140

g1 = np.column_stack([
    rng.normal(1750, 170, n1),  # lluvia_mm
    rng.normal(24.5, 1.2, n1),  # temp_media_c
    rng.normal(6.4, 0.25, n1),  # ph_suelo
    rng.normal(5.2, 0.7, n1),   # materia_organica_pct
    rng.normal(700, 110, n1),   # altitud_m
    rng.normal(5, 1.7, n1)      # pendiente_pct
])

g2 = np.column_stack([
    rng.normal(1450, 160, n2),
    rng.normal(19.5, 1.0, n2),
    rng.normal(6.9, 0.22, n2),
    rng.normal(3.9, 0.6, n2),
    rng.normal(1550, 190, n2),
    rng.normal(8.5, 2.0, n2)
])

g3 = np.column_stack([
    rng.normal(1150, 150, n3),
    rng.normal(15.2, 1.0, n3),
    rng.normal(5.7, 0.25, n3),
    rng.normal(3.1, 0.5, n3),
    rng.normal(2450, 220, n3),
    rng.normal(15.5, 3.0, n3)
])

X = np.vstack([g1, g2, g3])
df = pd.DataFrame(X, columns=[
    "lluvia_mm", "temp_media_c", "ph_suelo",
    "materia_organica_pct", "altitud_m", "pendiente_pct"
])

print("Dimensiones del dataset:", df.shape)
display(df.head())

# ----------------------------
# 2) Normalización
# ----------------------------
# Distancias euclídeas sensibles a escala -> StandardScaler (media 0, var 1).
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# ----------------------------
# 3) Estimación de bandwidth
# ----------------------------
# estimate_bandwidth usa percentiles de distancias para sugerir el radio del kernel.
# - quantile ∈ (0,1): percentil de distancias; más bajo => ventanas pequeñas => más clusters.
# - sample_size: submuestra para acelerar el cálculo en datasets grandes.
# Ajusta quantile si ves “muchos” o “pocos” clústeres.

bandwidth = estimate_bandwidth(
    X_scaled,
    quantile=0.15,      # valor típico entre 0.1 y 0.3
    n_samples=300,      # submuestra para velocidad
    random_state=11
)

print(f"Bandwidth estimado: {bandwidth:.4f}")

# Si bandwidth es muy pequeño o muy grande, ajusta el quantile y re-ejecuta.

# ----------------------------
# 4) Ajuste de Mean-Shift
# ----------------------------
# bin_seeding=True: inicializa con una cuadrícula de bins para acelerar.
# cluster_all=True: etiqueta todos los puntos; si False, puntos alejados reciben -1.
ms = MeanShift(
    bandwidth=bandwidth,
    bin_seeding=True,
    cluster_all=True
)
labels = ms.fit_predict(X_scaled)  # ajusta el modelo y devuelve sus etiquetas de entrenamiento
centers_scaled = ms.cluster_centers_
n_clusters_ = len(np.unique(labels))

print(f"Número de clústeres detectados: {n_clusters_}")
df_ms = df.copy()
df_ms["cluster"] = labels
print("\nConteo por clúster:")
print(df_ms["cluster"].value_counts().sort_index())

# Opcional: calidad de la partición (si hay >=2 clústeres)
if n_clusters_ >= 2:
    sil = silhouette_score(X_scaled, labels)
    print(f"\nSilhouette (>=2 clústeres): {sil:.3f}")
else:
    print("\nSilhouette no aplicable (solo 1 clúster). Ajusta 'quantile' para más granularidad.")

# ----------------------------
# 5) Visualización 2D (PCA)
# ----------------------------
pca = PCA(n_components=2, random_state=11)
X_2d = pca.fit_transform(X_scaled)
centers_2d = pca.transform(centers_scaled)

plt.figure(figsize=(6,5))
for c in range(n_clusters_):
    plt.scatter(X_2d[labels==c, 0], X_2d[labels==c, 1], s=18, alpha=0.7, label=f"Cluster {c}")
plt.scatter(centers_2d[:,0], centers_2d[:,1], s=180, marker='X', edgecolor='k', label="Centros")
plt.title("Mean-Shift en 2D (PCA)")
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.legend()
plt.grid(True)
plt.show()

# ----------------------------
# 6) Interpretación de clústeres
# ----------------------------
# Convertimos centros a unidades originales para lectura directa.
centers_original = pd.DataFrame(
    scaler.inverse_transform(centers_scaled),
    columns=df.columns
).round(2)
centers_original.index = [f"Cluster {i}" for i in range(n_clusters_)]

print("\nCentros (unidades originales):")
display(centers_original)

perfil = df_ms.groupby("cluster").mean(numeric_only=True).round(2)
perfil.index = [f"Cluster {i}" for i in perfil.index]
print("\nPerfil de variables por clúster (media):")
display(perfil)

mu = df.mean()
sigma = df.std(ddof=0)
z_centers = ((centers_original - mu) / sigma).round(2)
print("\nZ-scores de centros (positivo = sobre promedio global):")
display(z_centers)

print("""
================= CÓMO LEER LOS RESULTADOS =================
1) Mean-Shift detecta clústeres a partir de la densidad. No fija k.
   El número de clústeres depende del 'bandwidth' (ventana del kernel).

2) Gráfica PCA:
   - Nubes separadas y centros (X) indican segmentos bien diferenciados.

3) Centros en unidades originales:
   - Permiten describir cada segmento con variables del dominio (lluvia, altitud, etc.).

4) Z-scores:
   - Identifican variables “salientes” por clúster (>> 0 o << 0).

5) Si devuelve 1 solo clúster:
   - Reduce 'quantile' (ej. 0.10) para un bandwidth más pequeño y mayor granularidad.

6) Costo computacional:
   - Mean-Shift escala mal con n grande. Para datos masivos, considerar mini-batch K-means,
     HDBSCAN o un pre-agrupamiento + Mean-Shift en centros.
=============================================================
""")

# ----------------------------
# 7) “Predicción” para nuevos puntos
# ----------------------------
# MeanShift.predict asigna cada punto al centro aprendido más cercano.
# Aplicamos el MISMO scaler del ajuste; no lo volvemos a aprender.
# Reentrenar con más observaciones sería un nuevo ajuste, con otros centros posibles.

nuevo = pd.DataFrame([{
    "lluvia_mm": 1380,
    "temp_media_c": 18.8,
    "ph_suelo": 6.85,
    "materia_organica_pct": 3.6,
    "altitud_m": 1620,
    "pendiente_pct": 7.5
}])

asignacion = ms.predict(scaler.transform(nuevo))[0]
print("Nuevo ejemplo:")
display(nuevo)
print(f"Asignado al centro aprendido más cercano: Cluster {asignacion}")

print("""
Interpretación:
- Compara el nuevo registro con el centro correspondiente y con el perfil del grupo.
  Un centro de Mean-Shift no tiene por qué coincidir con la media de todo el grupo asignado.
- Esta asignación describe la simulación con centros fijos; no valida decisiones agronómicas.
  Reentrenar al añadir datos puede cambiar centros y grupos: sería otro modelo.
""")

# ----------------------------
# 8) Hiperparámetros clave para ajustar
# ----------------------------
print("""
Sugerencias de tuning:
- estimate_bandwidth.quantile: 0.10–0.30 suele ser buen rango. Más bajo => más clústeres.
- MeanShift(bin_seeding=True): acelera en alta dimensión.
- Aquí estandarizamos porque las unidades difieren; la elección de escala define qué similitud medimos.
- Si el dataset es muy grande: muestrear para estimar bandwidth o considerar alternativas.
""")
